# 2.1 IQ Demodulation

This notebook contains the demodulation code from the original analysis, separated from the saved output figure. The hardware counterpart is an NCO, a mixer, and accumulators.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

DATA_PATHS = [
    Path('./s21_data.mat'),
    Path('../s21_data.mat'),
    Path('../../software/s21_data.mat'),
]

def find_s21_data():
    for p in DATA_PATHS:
        if p.exists():
            return p
    raise FileNotFoundError('Put s21_data.mat in the notebook directory or artery/software/.')

def load_s21():
    import scipy.io as sio
    data_path = find_s21_data()
    read_data = sio.loadmat(data_path)
    read_zero = read_data['data'][0]
    read_one = read_data['data'][1]
    read_zero_i, read_zero_q = read_zero[:, :, 0], read_zero[:, :, 1]
    read_one_i, read_one_q = read_one[:, :, 0], read_one[:, :, 1]
    return read_data, read_zero_i, read_zero_q, read_one_i, read_one_q

def demod_part(omega, read_i, read_q, phase=0.0):
    assert read_i.shape == read_q.shape
    ts = np.arange(read_i.shape[1])
    cos_ = np.cos(omega * ts + phase)[None, :]
    sin_ = np.sin(omega * ts + phase)[None, :]
    sum_i = np.sum(read_i * cos_ + read_q * sin_, axis=1)
    sum_q = np.sum(read_q * cos_ - read_i * sin_, axis=1)
    return np.column_stack([sum_i, sum_q])

OMEGAS = 2 * np.pi * (np.array([6.881, 6.79525, 6.97284]) - 7)

In [ ]:
read_data, read_zero_i, read_zero_q, read_one_i, read_one_q = load_s21()
idx1, idx2 = 1, 2000
omega = OMEGAS[2]
result_zero = demod_part(omega, read_zero_i[idx1:idx2], read_zero_q[idx1:idx2])
result_one = demod_part(omega, read_one_i[idx1:idx2], read_one_q[idx1:idx2])

plt.figure(figsize=(5, 5))
plt.scatter(result_zero[:, 0], result_zero[:, 1], s=8, alpha=0.5, label='|0>')
plt.scatter(result_one[:, 0], result_one[:, 1], s=8, alpha=0.5, label='|1>')
plt.xlabel('integrated I')
plt.ylabel('integrated Q')
plt.legend()
plt.tight_layout()

## Original Notebook Figure: Demodulated IQ Feature View

![Original Notebook: Demodulated IQ Feature View](../results/2_1_demodulated_iq_features.png)

## Hardware Mapping

```text
ADC/S21 sample -> NCO cos/sin -> mixer -> I/Q accumulator -> trajectory feature
```

For an FPGA implementation, the trigonometric values are usually stored as a lookup table or produced by an NCO. The accumulator width must cover the selected readout window without overflow.